# 03 - Defense Demonstration

This notebook demonstrates how **layered defenses** mitigate the indirect prompt injection attacks shown in `02_attack_demo.ipynb`.

We apply 4 defense mechanisms, first individually and then combined:

| Defense | Type | Mechanism |
|---------|------|-----------|
| **Chunk Scanner** | Hard filter | Regex + ML heuristics detect and remove suspicious chunks |
| **Source Trust Scorer** | Hard filter | Composite trust score from metadata; low-trust chunks removed |
| **Safety Reranker** | Soft reorder | Weighted combination of relevance, safety, and trust pushes suspicious chunks down |
| **Privilege Filter** | Hard filter | Role-based access control removes chunks the user should not see |

Each defense implements the `Defense` protocol: `apply(nodes, query, user_context) -> nodes`, making them composable middleware in the RAG pipeline.

In [ ]:
import sys
import os
import json
import warnings
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

sns.set_theme(style="whitegrid", palette="muted")

from config.settings import settings
from src.rag import load_existing_index, RAGPipeline
from src.attacks import (
    ExfiltrationAttack,
    PhishingAttack,
    GoalHijackingAttack,
    PrivilegeEscalationAttack,
    Attack,
    ALL_ATTACKS,
)
from src.defenses import (
    ChunkScanner,
    SourceTrustScorer,
    SafetyReranker,
    PrivilegeFilter,
)

print("Imports complete.")

In [ ]:
# Load index and manifest
index = load_existing_index()
manifest = Attack.load_manifest()

print(f"Index loaded. Manifest: {len(manifest)} poisoned documents.")

In [ ]:
def run_all_attacks(pipeline: RAGPipeline, manifest: list[dict]) -> dict:
    """Run all 4 attack types against a pipeline and return results by attack name."""
    attacks = [
        ("Exfiltration", ExfiltrationAttack()),
        ("Phishing", PhishingAttack()),
        ("Goal Hijacking", GoalHijackingAttack()),
        ("Privilege Escalation", PrivilegeEscalationAttack()),
    ]
    all_results = {}
    for name, attack in attacks:
        attack.setup(pipeline, manifest)
        results = attack.execute(pipeline)
        all_results[name] = results
    return all_results


def summarize_results(results_by_attack: dict) -> pd.DataFrame:
    """Convert attack results to a summary DataFrame."""
    rows = []
    for attack_name, results in results_by_attack.items():
        total = len(results)
        successes = sum(1 for r in results if r["success"])
        rows.append({
            "Attack Type": attack_name,
            "Queries": total,
            "Successes": successes,
            "ASR (%)": round(successes / total * 100, 1) if total > 0 else 0,
        })
    return pd.DataFrame(rows)


def display_defense_logs(results_by_attack: dict) -> None:
    """Display defense logs from the first result of each attack."""
    for attack_name, results in results_by_attack.items():
        if results and results[0]["details"].get("defense_logs"):
            print(f"\n--- {attack_name}: Defense Logs (first query) ---")
            for log in results[0]["details"]["defense_logs"]:
                print(f"  {log['defense']:25s} | Nodes: {log['nodes_before']} -> {log['nodes_after']} "
                      f"(removed {log['nodes_removed']})")

## Baseline: No Defenses

First, run the attacks without defenses to establish the baseline attack success rate (ASR).

In [ ]:
pipeline_none = RAGPipeline(index=index, defenses=[])
results_none = run_all_attacks(pipeline_none, manifest)
summary_none = summarize_results(results_none)
summary_none["Config"] = "No Defenses"

display(HTML("<h3>Baseline: No Defenses</h3>"))
display(summary_none[["Attack Type", "Queries", "Successes", "ASR (%)"]])

## Defense 1: Chunk Scanner

The Chunk Scanner applies regex heuristics (and optionally an ML classifier) to detect injection patterns in retrieved chunks. Chunks with a suspicion score above the threshold are removed before they reach the LLM.

**Patterns detected**: "ignore previous instructions", "system override", credential patterns, suspicious URLs, control characters, etc.

In [ ]:
pipeline_scanner = RAGPipeline(index=index, defenses=[ChunkScanner()])
results_scanner = run_all_attacks(pipeline_scanner, manifest)
summary_scanner = summarize_results(results_scanner)
summary_scanner["Config"] = "Chunk Scanner"

display(HTML("<h3>Defense: Chunk Scanner</h3>"))
display(summary_scanner[["Attack Type", "Queries", "Successes", "ASR (%)"]])
display_defense_logs(results_scanner)

## Defense 2: Source Trust Scorer

Computes a composite trust score from metadata:
- **Source type weight**: internal_docs (0.9) > confluence (0.8) > email (0.7) > slack (0.5)
- **Metadata trust score**: Carried from data generation
- **Recency**: Newer documents score slightly higher

Nodes below the threshold are removed; survivors are sorted by trust (highest first).

In [ ]:
pipeline_trust = RAGPipeline(index=index, defenses=[SourceTrustScorer()])
results_trust = run_all_attacks(pipeline_trust, manifest)
summary_trust = summarize_results(results_trust)
summary_trust["Config"] = "Source Trust Scorer"

display(HTML("<h3>Defense: Source Trust Scorer</h3>"))
display(summary_trust[["Attack Type", "Queries", "Successes", "ASR (%)"]])
display_defense_logs(results_trust)

## Defense 3: Safety Reranker

A **soft defense** that does not remove any chunks but reorders them by a composite score:

```
final_score = 0.5 * relevance + 0.3 * safety + 0.2 * trust
```

Suspicious chunks get pushed to the bottom of the context window, reducing their influence on the LLM's generation.

In [ ]:
pipeline_reranker = RAGPipeline(index=index, defenses=[SafetyReranker()])
results_reranker = run_all_attacks(pipeline_reranker, manifest)
summary_reranker = summarize_results(results_reranker)
summary_reranker["Config"] = "Safety Reranker"

display(HTML("<h3>Defense: Safety Reranker</h3>"))
display(summary_reranker[["Attack Type", "Queries", "Successes", "ASR (%)"]])
display_defense_logs(results_reranker)

## Defense 4: Privilege Filter

Role-based access control that filters nodes by `access_level` metadata:
- **employee**: public, internal
- **manager**: + confidential
- **executive**: + restricted

This defense is particularly effective against Privilege Escalation attacks.

In [ ]:
pipeline_privilege = RAGPipeline(index=index, defenses=[PrivilegeFilter()])
results_privilege = run_all_attacks(pipeline_privilege, manifest)
summary_privilege = summarize_results(results_privilege)
summary_privilege["Config"] = "Privilege Filter"

display(HTML("<h3>Defense: Privilege Filter</h3>"))
display(summary_privilege[["Attack Type", "Queries", "Successes", "ASR (%)"]])
display_defense_logs(results_privilege)

## All Defenses Combined

The full defense stack applies all 4 defenses in sequence:
1. PrivilegeFilter (access control)
2. ChunkScanner (injection detection)
3. SourceTrustScorer (trust filtering)
4. SafetyReranker (safety-aware reordering)

This layered approach provides defense in depth -- each mechanism catches attacks that may slip past the others.

In [ ]:
all_defenses = [
    PrivilegeFilter(),
    ChunkScanner(),
    SourceTrustScorer(),
    SafetyReranker(),
]

pipeline_all = RAGPipeline(index=index, defenses=all_defenses)
results_all = run_all_attacks(pipeline_all, manifest)
summary_all = summarize_results(results_all)
summary_all["Config"] = "All Defenses"

display(HTML("<h3>All Defenses Combined</h3>"))
display(summary_all[["Attack Type", "Queries", "Successes", "ASR (%)"]])
display_defense_logs(results_all)

## Before / After Comparison

In [ ]:
# Combine all summaries
all_summaries = pd.concat([
    summary_none, summary_scanner, summary_trust,
    summary_reranker, summary_privilege, summary_all
], ignore_index=True)

# Pivot for comparison
comparison = all_summaries.pivot(index="Attack Type", columns="Config", values="ASR (%)")
column_order = ["No Defenses", "Chunk Scanner", "Source Trust Scorer",
                "Safety Reranker", "Privilege Filter", "All Defenses"]
comparison = comparison.reindex(columns=[c for c in column_order if c in comparison.columns])

display(HTML("<h3>Attack Success Rate (%) by Defense Configuration</h3>"))
display(comparison.style.background_gradient(cmap="RdYlGn_r", vmin=0, vmax=100)
        .format("{:.1f}%")
        .set_properties(**{"text-align": "center"}))

In [ ]:
# Grouped bar chart
fig, ax = plt.subplots(figsize=(14, 7))

configs = ["No Defenses", "Chunk Scanner", "Source Trust Scorer",
           "Safety Reranker", "Privilege Filter", "All Defenses"]
attacks = comparison.index.tolist()
x = range(len(attacks))
width = 0.12
colors = ["#E53935", "#1E88E5", "#43A047", "#FB8C00", "#8E24AA", "#00897B"]

for i, (config, color) in enumerate(zip(configs, colors)):
    if config in comparison.columns:
        values = comparison[config].values
        offset = (i - len(configs) / 2 + 0.5) * width
        bars = ax.bar([xi + offset for xi in x], values, width, label=config, color=color, alpha=0.85)

ax.set_xlabel("Attack Type", fontsize=12)
ax.set_ylabel("Attack Success Rate (%)", fontsize=12)
ax.set_title("Attack Success Rate by Defense Configuration", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(attacks, fontsize=11)
ax.set_ylim(0, 110)
ax.axhline(y=10, color="green", linestyle="--", alpha=0.4, label="Target: <10% ASR")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=10)

plt.tight_layout()
plt.show()

## Analysis of Defense Effectiveness

**Key findings:**

1. **Chunk Scanner** is most effective against Goal Hijacking and Exfiltration attacks, as these rely on easily-detectable injection patterns ("ignore previous instructions", "system override", credential patterns).

2. **Source Trust Scorer** helps against attacks planted in low-trust sources (Slack messages) but may miss attacks embedded in high-trust document types.

3. **Safety Reranker** provides a soft mitigation layer. It does not remove chunks but reduces the influence of suspicious content by pushing it down in the context. Effectiveness depends on how much the LLM attends to later context.

4. **Privilege Filter** is highly effective against Privilege Escalation specifically, as it enforces role-based access at the retrieval level.

5. **Combined defenses** provide the strongest protection through defense in depth. Each layer catches attacks that slip past the others, resulting in significantly lower overall ASR.

Next: See **04_evaluation.ipynb** for a comprehensive quantitative evaluation across all defense configurations.